# BuzzyHive 2.0 — KNN Model Training

Generates a synthetic dataset from reference sensor distributions (Aida 'Izwani, 2025),
trains a KNN classifier for kelulut honey harvest readiness, and saves `model.pkl` + `scaler.pkl`.

**Classes:** `not_ready` · `approaching` · `nearly_ready` · `ready`  
**Features:** `mq2_value`, `mq3_value`, `mq5_value`, `mq135_value`, `temp`, `humidity`

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
BASE = Path('.')

## 1. Synthetic Dataset Generation

Anchor points from Sheet2 of `reference kelulut Taufiq.xlsx` (3 classes).
Class 4 (`ready`) is extrapolated by continuing the linear trend across sensors.
Temperature and humidity follow plausible kelulut hive ranges per readiness stage.
50 samples per class → 200 total.

In [ ]:
# (mean, std) per feature per class
# MQ values: raw ADC units — sourced from Sheet2 reference data
# temp (°C) and humidity (%) : plausible kelulut hive ranges
ANCHORS = {
    'not_ready':    {'mq2': (43,  4), 'mq3': (38,  4), 'mq5': (112, 5), 'mq135': (160, 6), 'temp': (28.5, 1.0), 'humidity': (66.0, 3.0)},
    'approaching':  {'mq2': (81,  4), 'mq3': (83,  4), 'mq5': (150, 5), 'mq135': (191, 6), 'temp': (29.5, 1.0), 'humidity': (69.0, 3.0)},
    'nearly_ready': {'mq2': (119, 4), 'mq3': (129, 4), 'mq5': (200, 5), 'mq135': (239, 6), 'temp': (30.5, 1.0), 'humidity': (72.0, 3.0)},
    'ready':        {'mq2': (157, 4), 'mq3': (175, 4), 'mq5': (244, 5), 'mq135': (279, 6), 'temp': (31.5, 1.0), 'humidity': (75.0, 3.0)},
}

SAMPLES_PER_CLASS = 50
FEATURES = ['mq2_value', 'mq3_value', 'mq5_value', 'mq135_value', 'temp', 'humidity']

rows = []
for label, p in ANCHORS.items():
    for _ in range(SAMPLES_PER_CLASS):
        rows.append({
            'mq2_value':   max(0, int(round(np.random.normal(p['mq2'][0],   p['mq2'][1])))),
            'mq3_value':   max(0, int(round(np.random.normal(p['mq3'][0],   p['mq3'][1])))),
            'mq5_value':   max(0, int(round(np.random.normal(p['mq5'][0],   p['mq5'][1])))),
            'mq135_value': max(0, int(round(np.random.normal(p['mq135'][0], p['mq135'][1])))),
            'temp':        round(np.random.normal(p['temp'][0], p['temp'][1]), 1),
            'humidity':    round(float(np.clip(np.random.normal(p['humidity'][0], p['humidity'][1]), 0, 100)), 1),
            'label': label,
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
df.to_csv(BASE / 'dataset.csv', index=False)

print(f"Dataset: {df.shape[0]} samples, {df.shape[1]-1} features")
df.groupby('label').size()

In [ ]:
df.describe().round(2)

## 2. Train / Test Split + Scaling

In [ ]:
X = df[FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

## 3. Find Best K via 5-Fold Cross-Validation

In [ ]:
k_range = range(1, 16)
cv_scores = [
    cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=5).mean()
    for k in k_range
]

best_k = list(k_range)[int(np.argmax(cv_scores))]
print(f"Best K = {best_k}  |  CV accuracy = {max(cv_scores):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), cv_scores, 'o-', color='steelblue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K')
plt.ylabel('CV Accuracy')
plt.title('KNN — Cross-Validation Accuracy vs K')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Train Final Model + Evaluate

In [ ]:
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

CLASS_ORDER = ['not_ready', 'approaching', 'nearly_ready', 'ready']
print(classification_report(y_test, y_pred, target_names=CLASS_ORDER))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix — BuzzyHive KNN')
plt.tight_layout()
plt.show()

## 5. Save Model + Scaler

In [ ]:
pickle.dump(model,  open(BASE / 'model.pkl',  'wb'))
pickle.dump(scaler, open(BASE / 'scaler.pkl', 'wb'))

print(f"Saved model.pkl  (K={best_k}, classes={model.classes_.tolist()})")
print(f"Saved scaler.pkl (features={FEATURES})")

## 6. Quick Inference Test
Verify the saved model returns the expected shape before connecting to Laravel.

In [ ]:
HRI_MAP = {'not_ready': 0.25, 'approaching': 0.50, 'nearly_ready': 0.75, 'ready': 1.00}

test_samples = {
    'not_ready sample':    [43,  38,  112, 160, 28.5, 66.0],
    'approaching sample':  [81,  83,  150, 191, 29.5, 69.0],
    'nearly_ready sample': [119, 129, 200, 239, 30.5, 72.0],
    'ready sample':        [157, 175, 244, 279, 31.5, 75.0],
}

m = pickle.load(open(BASE / 'model.pkl',  'rb'))
s = pickle.load(open(BASE / 'scaler.pkl', 'rb'))

print(f"{'Sample':<25} {'Predicted':<15} {'hri_value':<12} {'confidence'}")
print('-' * 65)
for name, values in test_samples.items():
    X_in = s.transform([values])
    label = m.predict(X_in)[0]
    conf  = round(max(m.predict_proba(X_in)[0]), 4)
    print(f"{name:<25} {label:<15} {HRI_MAP[label]:<12} {conf}")